In [ ]:
# @title
pip install pytorch-msssim

In [ ]:
# @title
import os
import zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from pytorch_msssim import ssim

In [ ]:
# @title
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title
import os

file_path = '/content/drive/MyDrive/Colab Notebooks/Under Water Image Restoration with UNET'

for item in os.listdir(file_path):
    print(item)

# Extract Zip File

In [ ]:
# @title
zip_path='/content/drive/MyDrive/Colab Notebooks/Under Water Image Restoration with UNET/Data/Paired.zip'
extract_path = '/content/dataset'
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
    print("Dataset Unzipped successfully!")

# Paths

In [ ]:
# @title
base_path="/content/dataset/Paired"

subsets = [
    "underwater_scenes",
    "underwater_imagenet",
    "underwater_dark"
]

print(os.listdir(base_path))


In [ ]:
# @title


In [ ]:
# @title
for subset in subsets:
    subset_path = os.path.join(base_path, subset)

    print(f"\n{subset}")
    print(os.listdir(subset_path))

# Drop TXT  File

In [ ]:
# @title
base_dir = Path("/content/dataset/Paired")

for file_path in base_dir.rglob("*"):
    if file_path.is_file() and file_path.suffix.lower() == ".txt":
        file_path.unlink()
        print(f"Deleted: {file_path}")

print("All text files removed successfully.")

# Check Corupted Images

In [ ]:
# @title
def check_corrupted_images(base_path="/content/dataset/Paired"):
    base_dir = Path(base_path)
    subsets = ["underwater_scenes", "underwater_imagenet", "underwater_dark"]
    valid_extensions = {".jpg", ".jpeg"}

    print("=== Starting Corruption Check Across All Subsets ===")

    for subset_name in subsets:
        subset_dir = base_dir / subset_name
        if not subset_dir.exists():
            continue

        for split_dir_name in ["trainA", "trainB", "valA", "valB", "validation"]:
            split_dir = subset_dir / split_dir_name
            if not split_dir.exists():
                continue

            image_files = [f for f in split_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_extensions]
            corrupt_count = 0

            for img_path in image_files:
                try:
                    with Image.open(img_path) as img:
                        img.verify()
                except Exception:
                    corrupt_count += 1
                    print(f"[CORRUPT FOUND] {img_path}")

            print(f"Checked {subset_name}/{split_dir_name}: {len(image_files)} files scanned, {corrupt_count} corrupt.")

    print("=== Corruption Check Completed ===")

check_corrupted_images()

In [ ]:
# @title


# Check Pair

In [ ]:
# @title
from pathlib import Path

def verify_files_flexible(base_path="/content/dataset/Paired"):
    base_dir = Path(base_path)
    subsets = ["underwater_scenes", "underwater_imagenet", "underwater_dark"]

    for subset in subsets:
        trainA = base_dir / subset / "trainA"
        trainB = base_dir / subset / "trainB"

        if not trainA.exists() or not trainB.exists():
            continue

        files_a = sorted([f.name for f in trainA.iterdir() if f.is_file() and f.suffix.lower() != ".txt"])
        files_b = sorted([f.name for f in trainB.iterdir() if f.is_file() and f.suffix.lower() != ".txt"])

        print(f"Subset: {subset}")
        print(f"TrainA: {len(files_a)} files | TrainB: {len(files_b)} files")

        if files_a == files_b:
            print("Status: Pass (Filenames match perfectly)\n")
        else:
            print("Status: Fail (Filenames do not match)\n")

verify_files_flexible()

# DataSet and DataLoader

In [ ]:
# @title
import os
from pathlib import Path
import random
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from PIL import Image

class EUVPDataset(Dataset):
    def __init__(self, base_path, subset_name, transform=None, is_train=True):
        self.subset_dir = Path(base_path) / subset_name
        self.transform = transform
        self.is_train = is_train

        self.A_dir = self.subset_dir / "trainA"
        self.B_dir = self.subset_dir / "trainB"

        #Just Pick Valid Images
        self.image_filenames = sorted([
            f.name for f in self.A_dir.iterdir()
            if f.is_file() and f.suffix.lower() in {".jpg", ".jpeg", ".png"}
        ])

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_name = self.image_filenames[idx]

        img_a_path = self.A_dir / img_name
        img_b_path = self.B_dir / img_name

        image_a = Image.open(img_a_path).convert("RGB")
        image_b = Image.open(img_b_path).convert("RGB")

        if self.transform:
            image_a = self.transform(image_a)
            image_b = self.transform(image_b)

        # Data Augmentation: Random Flips
        if self.is_train:
            # 50% chance horizontal flip
            if random.random() > 0.5:
                image_a = torch.flip(image_a, dims=[2])
                image_b = torch.flip(image_b, dims=[2])

            # 50% chance vertical flip
            if random.random() > 0.5:
                image_a = torch.flip(image_a, dims=[1])
                image_b = torch.flip(image_b, dims=[1])

        return image_a, image_b

# 1. Transforms
transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
])

BASE_PATH = "/content/dataset/Paired"
subsets = ["underwater_scenes", "underwater_imagenet", "underwater_dark"]

# 2. Dataset for all three subset
dataset_list = []
for subset in subsets:
    ds = EUVPDataset(base_path=BASE_PATH, subset_name=subset, transform=transform, is_train=True)
    dataset_list.append(ds)
    print(f"Loaded {subset}: {len(ds)} samples")

# ConcatDataset all subset
combined_dataset = ConcatDataset(dataset_list)
print(f"\nTotal Combined Dataset Samples: {len(combined_dataset)}")

# 3. Initialize DataLoader
dataloader = DataLoader(combined_dataset, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)

# Batch testing
for batch_a, batch_b in dataloader:
    print("Combined Batch A (Degraded) shape:", batch_a.shape)
    print("Combined Batch B (Reference) shape:", batch_b.shape)
    break

# Model

In [ ]:
# @title
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    """(Conv2D -> BatchNorm -> ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()

        # Encoder (Downsampling)
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))

        # Decoder (Upsampling + Skip Connections)
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(1024, 512)

        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256, 128)

        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(128, 64)

        # Final Output Layer
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Encoder path
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        # Decoder path with skip connections
        x = self.up1(x5)
        x = torch.cat([x, x4], dim=1)
        x = self.conv1(x)

        x = self.up2(x)
        x = torch.cat([x, x3], dim=1)
        x = self.conv2(x)

        x = self.up3(x)
        x = torch.cat([x, x2], dim=1)
        x = self.conv3(x)

        x = self.up4(x)
        x = torch.cat([x, x1], dim=1)
        x = self.conv4(x)

        logits = self.outc(x)
        return self.sigmoid(logits)  # [0, 1] for Sigmoid

# Model testing with a dummy batch from our dataloader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(in_channels=3, out_channels=3).to(device)

# Forward Patch for Sample
for sample_a, sample_b in dataloader:
    sample_a = sample_a.to(device)
    output = model(sample_a)
    print("Model Output Shape:", output.shape)
    break

# Combined Loss

In [ ]:
# @title
import torch
import torch.nn as nn

class VibrantColorEdgeLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1_loss = nn.L1Loss()

    def forward(self, pred, target):
        # 1. Standard L1 Loss for overall structure
        loss_l1 = self.l1_loss(pred, target)

        # 2. Corrected RGB Channel Color Penalties (Red, Green, Blue channels explicitly defined)
        loss_red = torch.mean(torch.abs(pred[:, 0, :, :] - target[:, 0, :, :]))     # Red channel (crucial for corals/fish)
        loss_green = torch.mean(torch.abs(pred[:, 1, :, :] - target[:, 1, :, :])) # Green channel
        loss_blue = torch.mean(torch.abs(pred[:, 2, :, :] - target[:, 2, :, :]))  # Blue channel

        loss_color = loss_red + 0.5 * loss_green + 0.5 * loss_blue

        # 3. Edge / Gradient Loss ( For Sharpness and High Resolution)
        pred_dx = torch.abs(pred[:, :, :, :-1] - pred[:, :, :, 1:])
        pred_dy = torch.abs(pred[:, :, :-1, :] - pred[:, :, 1:, :])

        target_dx = torch.abs(target[:, :, :, :-1] - target[:, :, :, 1:])
        target_dy = torch.abs(target[:, :, :-1, :] - target[:, :, 1:, :])

        loss_edge = torch.mean(torch.abs(pred_dx - target_dx)) + torch.mean(torch.abs(pred_dy - target_dy))

        # Total Loss: L1 + Strong Color Weight (0.5) + Edge Weight (0.4)
        loss_total = loss_l1 + 0.5 * loss_color + 0.4 * loss_edge
        return loss_total

#Train

In [ ]:
# @title


In [ ]:
# @title
import os
import torch
import torch.optim as optim

# 1. Create Folder on Google Drive
project_dir = "/content/drive/MyDrive/Colab Notebooks/Under Water Image Restoration with UNET"
os.makedirs(project_dir, exist_ok=True)
print(f"Project directory ready: {project_dir}")

# 2. Device Agnostics
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. Model, Loss Function and Optimizer initializition
model = UNet(in_channels=3, out_channels=3).to(device)

criterion = VibrantColorEdgeLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0002)

# 4. Paths for checkpoint and final model inside the project folder
checkpoint_path = os.path.join(project_dir, "under_water_image_restoration_checkpoint.pth")
final_model_path = os.path.join(project_dir, "under_water_image_restoration.pth")

# --- FRESH START SETTING ---
start_fresh = True
if start_fresh and os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
    print("Deletd Old Checkpoints")

start_epoch = 0
num_epochs = 15

if not start_fresh and os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f"Checkpoint loaded! Resuming training from Epoch {start_epoch + 1}/{num_epochs}")
else:
    print("Starting training fresh from Epoch 1 with VibrantColorEdgeLoss...")

scaler = torch.amp.GradScaler('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_grad_enabled(True)

# 5. Training Loop
for epoch in range(start_epoch, num_epochs):
    model.train()
    running_loss = 0.0

    for batch_idx, (images_a, images_b) in enumerate(dataloader):
        images_a = images_a.to(device)
        images_b = images_b.to(device)

        optimizer.zero_grad()

        # Automatic Mixed Precision (AMP)
        with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
            outputs = model(images_a)
            loss = criterion(outputs, images_b)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        if (batch_idx + 1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{batch_idx+1}/{len(dataloader)}], Loss: {loss.item():.4f}")

    epoch_loss = running_loss / len(dataloader)
    print(f"=== Epoch [{epoch+1}/{num_epochs}] Completed | Average Loss: {epoch_loss:.4f} ===\n")

    # Checkpoint save inside Google Drive project folder
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }
    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved for Epoch {epoch+1} in project folder.\n")

# Final Model save inside Google Drive project folder
torch.save(model.state_dict(), final_model_path)
print(f"Training completed successfully! Final model saved as: {final_model_path}")

In [ ]:
# @title
import matplotlib.pyplot as plt
import torch

# Model in eval mode
model.eval()

# Pick Batch from DataLoader
for images_a, images_b in dataloader:
    images_a_gpu = images_a.to(device)

    with torch.no_grad():
        outputs = model(images_a_gpu)

    # Send Outputs to CPU
    outputs = outputs.cpu()

    # Show only 4 Images
    fig, axes = plt.subplots(4, 3, figsize=(12, 16))

    for i in range(4):
        # 1. Degraded Image (Input)
        img_input = images_a[i].permute(1, 2, 0).numpy()

        # 2. Restored Image
        img_out = outputs[i].permute(1, 2, 0).numpy()
        img_out = (img_out + 1) / 2 if img_out.min() < 0 else img_out
        img_out = img_out.clip(0, 1)

        # 3. Reference Image
        img_ref = images_b[i].permute(1, 2, 0).numpy()

        axes[i, 0].imshow(img_input.clip(0, 1))
        axes[i, 0].set_title("Degraded (Input)")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(img_out)
        axes[i, 1].set_title("Restored (Output)")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(img_ref.clip(0, 1))
        axes[i, 2].set_title("Reference (Ground Truth)")
        axes[i, 2].axis("off")

    plt.tight_layout()
    plt.show()
    break

In [ ]:
# @title
